In [1]:
import pandas as pd

df_train = pd.read_csv("train.csv")
df_test = pd.read_csv("test.csv")

df_train.head()

,id,gas_raw,temp_c,flame,gas_avg,temp_avg,dGas,dTemp,sensor_noise,timestamp_ms,sensor_id,battery_level,vehicle_speed,weather_condition,gas_sensor_drift,ambient_temp,label
0,7621,422.194432,118.043366,1,767.789333,NaN,-55.906124,2.801129,NaN,1212189282,S001,NaN,123.645528,rainy,362.321329,102.058677,FIRE
1,6224,652.020520,121.762018,1,-116.321698,117.187637,-25.912647,1.523781,-0.446243,1948644533,S003,94.640768,72.397748,rainy,518.720369,123.267611,FIRE
2,6584,248.605877,6.377835,0,275.077483,NaN,NaN,NaN,-0.920522,1403281804,S004,85.680628,87.808526,sunny,289.947889,0.763587,NORMAL
3,7845,872.419262,111.909726,1,865.917087,234.145157,-31.096726,0.779816,1.205554,1969672422,S001,102.969677,115.677446,rainy,927.954621,114.443297,FIRE
4,6567,1919.594960,NaN,0,NaN,NaN,NaN,-1.301399,-0.779358,1844367040,S004,NaN,71.741976,sunny,2193.352397,NaN,GAS_LEAK


In [2]:
df_train.label.unique()

array(['FIRE', 'NORMAL', 'GAS_LEAK', 'OVERHEAT'], dtype=object)

In [3]:
#subtask1
vals = ['NORMAL', 'GAS_LEAK', 'OVERHEAT', 'FIRE']
procs = []
temps = []
cons = []
for val in vals:
    proc = round(df_train.label.value_counts()[val] / len(df_train.label) * 100, 2)
    procs.append(proc)
    temp = round(df_train[df_train.label == val].temp_c.mean(), 1)
    temps.append(temp)
    con = round(df_train[df_train.label == val].gas_raw.mean(), 1)
    cons.append(con)

df_1 = pd.DataFrame({
    "subtaskID": 1,
    "datapointID": range(1, 13),
    "answer": procs + temps + cons
})

df_1

,subtaskID,datapointID,answer
0,1,1,25.12
1,1,2,24.94
2,1,3,24.72
3,1,4,25.22
4,1,5,28.50
5,1,6,46.90
6,1,7,93.00
7,1,8,112.00
8,1,9,321.20
9,1,10,867.40


In [4]:
features = [col for col in df_test.columns if col != 'id' and col != 'weather_condition' and col != 'sensor_id']
target = 'label'

X = df_train[features]
y = df_train[target]

X_final = df_test[features]

for col in features:
    median = X[col].median()
    X[col] = X[col].fillna(median)
    X_final[col] = X_final[col].fillna(median)

X.head()

C:\Users\Ilie-Goga\AppData\Local\Temp\ipykernel_19848\1926768452.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X[col] = X[col].fillna(median)
C:\Users\Ilie-Goga\AppData\Local\Temp\ipykernel_19848\1926768452.py:12: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X_final[col] = X_final[col].fillna(median)
C:\Users\Ilie-Goga\AppData\Local\Temp\ipykernel_19848\1926768452.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer

,gas_raw,temp_c,flame,gas_avg,temp_avg,dGas,dTemp,sensor_noise,timestamp_ms,battery_level,vehicle_speed,gas_sensor_drift,ambient_temp
0,422.194432,118.043366,1,767.789333,63.636718,-55.906124,2.801129,0.088936,1212189282,60.006369,123.645528,362.321329,102.058677
1,652.020520,121.762018,1,-116.321698,117.187637,-25.912647,1.523781,-0.446243,1948644533,94.640768,72.397748,518.720369,123.267611
2,248.605877,6.377835,0,275.077483,63.636718,1.106763,-0.004388,-0.920522,1403281804,85.680628,87.808526,289.947889,0.763587
3,872.419262,111.909726,1,865.917087,234.145157,-31.096726,0.779816,1.205554,1969672422,102.969677,115.677446,927.954621,114.443297
4,1919.594960,73.652402,0,454.164968,63.636718,1.106763,-1.301399,-0.779358,1844367040,60.006369,71.741976,2193.352397,69.570978


In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

model = RandomForestClassifier()

cross_val_score(model, X, y, cv=5, scoring='f1_weighted')

array([0.95836716, 0.96766147, 0.95837979, 0.9690527 , 0.95213012])

In [6]:
model.fit(X, y)
y_preds = model.predict(X_final)

In [7]:
df_4 = pd.DataFrame({
    "subtaskID": 4,
    "datapointID": df_test.id,
    "answer": y_preds
})

df_4.head()

,subtaskID,datapointID,answer
0,4,5178,OVERHEAT
1,4,8047,GAS_LEAK
2,4,5976,OVERHEAT
3,4,7812,FIRE
4,4,7487,FIRE


In [8]:
df_train.label = df_train.label.map({"NORMAL":0, "GAS_LEAK":1, "OVERHEAT":2, "FIRE":3})

In [9]:
sorted(df_train.corr(method="spearman", numeric_only=True)["label"])

[-0.013329531600247798,
 -0.011716472117709762,
 0.004951575306987509,
 0.006830161227494715,
 0.019027615020524698,
 0.02277714918225236,
 0.023150219422934763,
 0.3592780759814339,
 0.36111616522065915,
 0.3778306658799599,
 0.7490067683865399,
 0.7593231280246343,
 0.7717269657334737,
 0.7783608634437618,
 1.0]

In [10]:
corr_cols = ["temp_avg", "flame", "temp_c", "ambient_temp", "gas_avg", "gas_sensor_drift"]
X = X[corr_cols]
X_final = X_final[corr_cols]

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X = pd.DataFrame(scaler.fit_transform(X), columns=corr_cols)
X_final = pd.DataFrame(scaler.transform(X_final), columns=corr_cols)

X.head()

,temp_avg,flame,temp_c,ambient_temp,gas_avg,gas_sensor_drift
0,-0.097837,1.733481,1.038824,0.705772,0.664051,-0.637759
1,1.104736,1.733481,1.121717,1.176145,-1.989695,-0.196957
2,-0.097837,-0.576874,-1.450335,-1.540755,-0.814872,-0.841739
3,3.731207,1.733481,0.902097,0.980439,0.958591,0.956446
4,-0.097837,-0.576874,0.049296,-0.014742,-0.277323,4.522895


In [13]:
X["label"] = df_train["label"]
X_norm = X[X.label == 0]
X_norm = X_norm.drop(columns="label")
X = X.drop(columns="label")
X.head()

,temp_avg,flame,temp_c,ambient_temp,gas_avg,gas_sensor_drift
0,-0.097837,1.733481,1.038824,0.705772,0.664051,-0.637759
1,1.104736,1.733481,1.121717,1.176145,-1.989695,-0.196957
2,-0.097837,-0.576874,-1.450335,-1.540755,-0.814872,-0.841739
3,3.731207,1.733481,0.902097,0.980439,0.958591,0.956446
4,-0.097837,-0.576874,0.049296,-0.014742,-0.277323,4.522895


In [15]:
import numpy as np
from scipy.spatial.distance import mahalanobis

mu = X_norm.mean().values
Sigma = np.cov(X_norm.values, rowvar=False)
Sigma_reg = Sigma + np.eye(Sigma.shape[0]) * 1e-6
Sigma_inv = np.linalg.inv(Sigma_reg)

scores = []
for i in range(len(X_final)):
    x = X_final.iloc[i].values
    d = mahalanobis(x, mu, Sigma_inv)
    scores.append(round(d, 2))

scores[:5]

[2.04, 2.49, 1.25, 2310.36, 2310.36]

In [16]:
df_2 = pd.DataFrame({
    "subtaskID": 2,
    "datapointID": df_test.id,
    "answer": scores
})

df_2.head()

,subtaskID,datapointID,answer
0,2,5178,2.04
1,2,8047,2.49
2,2,5976,1.25
3,2,7812,2310.36
4,2,7487,2310.36


In [20]:
p33 = pd.Series(scores).quantile(0.33)
p66 = pd.Series(scores).quantile(0.66)

def get_val(score):
    if score < p33:
        return "LOW"
    elif score < p66:
        return "MEDIUM"
    else:
        return "HIGH"

df_3 = pd.DataFrame({
    "subtaskID": 3,
    "datapointID": df_test.id,
    "answer": pd.Series(scores).apply(get_val)
})

df_3.head()

,subtaskID,datapointID,answer
0,3,5178,MEDIUM
1,3,8047,MEDIUM
2,3,5976,LOW
3,3,7812,HIGH
4,3,7487,HIGH


In [21]:
submission = pd.concat([df_1, df_2, df_3, df_4])
submission

,subtaskID,datapointID,answer
0,1,1,25.12
1,1,2,24.94
2,1,3,24.72
3,1,4,25.22
4,1,5,28.5
...,...,...,...
795,4,8600,FIRE
796,4,8402,GAS_LEAK
797,4,6744,OVERHEAT
798,4,7891,GAS_LEAK


In [22]:
submission.to_csv("submission.csv", index=False)